# Optional: CPU and WebGPU measurements

Use this notebook to check compiled logits and bounding boxes against eager
PyTorch and measure inference performance. The camera demos do not run these
comparisons.

Run once with `DEVICE = 'cpu'`, then with `DEVICE = 'webgpu'` in a fresh kernel.
Run one notebook at a time. Saved outputs come from a local browser run;
running it again measures your own device.


In [1]:
import statistics
import cv2
import json
import time
from pathlib import Path
from IPython.display import Markdown, display

import numpy as np
import torch
from IPython.display import Image as DisplayImage
import sys
sys.path.insert(0, str(Path('dfine_models.zip').resolve()))

from dfine_models.nn.backbone.hgnetv2 import HGNetv2
from dfine_models.zoo.dfine.dfine import DFINE
from dfine_models.zoo.dfine.dfine_decoder import DFINETransformer
from dfine_models.zoo.dfine.hybrid_encoder import HybridEncoder


DEVICE = 'webgpu'
IMAGE_PATH = 'YellowLabradorLooking_new.jpg'
SCORE_THRESHOLD = 0.35
classes = json.loads(Path('coco_classes.json').read_text())
torch.autograd.set_multithreading_enabled(False)
torch.manual_seed(11)
print('PyTorch:', torch.__version__, '| device:', DEVICE)

print('OpenCV:', cv2.__version__)


PyTorch: 2.15.0a0 | device: webgpu
OpenCV: 5.0.0


## 1. Load the model and sample

The first load downloads and verifies the original weights; later loads use the browser cache.


In [2]:
from dfine_weights import load_checkpoint

async def load_model():
    checkpoint = await load_checkpoint()
    model = DFINE(
        backbone=HGNetv2(name='B0', return_idx=[1, 2, 3], freeze_at=-1,
                        freeze_norm=False, use_lab=True, pretrained=False),
        encoder=HybridEncoder(in_channels=[256, 512, 1024], hidden_dim=256,
                              depth_mult=0.34, expansion=0.5,
                              eval_spatial_size=[640, 640]),
        decoder=DFINETransformer(feat_channels=[256, 256, 256],
                                num_layers=3, num_points=[3, 6, 3],
                                eval_spatial_size=[640, 640]),
    ).eval()
    model.load_state_dict(checkpoint['model'], strict=True)
    del checkpoint
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model


model = None
load_started = time.perf_counter()
model = await load_model()
weights_and_model_load_seconds = time.perf_counter() - load_started
print(f'Pretrained model ready: {sum(p.numel() for p in model.parameters()):,} parameters')

def prepare_frame(frame):
    height, width = frame.shape[:2]
    scale = min(640 / width, 640 / height)
    fitted_width = max(1, round(width * scale))
    fitted_height = max(1, round(height * scale))
    left, top = (640 - fitted_width) // 2, (640 - fitted_height) // 2
    canvas = np.full((640, 640, 3), 114, dtype=np.uint8)
    canvas[top:top + fitted_height, left:left + fitted_width] = cv2.resize(
        frame, (fitted_width, fitted_height), interpolation=cv2.INTER_LINEAR)
    rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB).astype(np.float32) / 255
    tensor = torch.from_numpy(rgb.transpose(2, 0, 1).copy()).unsqueeze(0)
    return tensor, (left, top, width / fitted_width, height / fitted_height)

photo = cv2.imread(IMAGE_PATH)
if photo is None:
    raise FileNotFoundError(f'Could not read {IMAGE_PATH}')
values, transform = prepare_frame(photo)
print('Model input:', tuple(values.shape))


Loading pretrained weights (first download: 39.9 MiB)...
Using verified cached weights.
Pretrained model ready: 10,321,877 parameters
Model input: (1, 3, 640, 640)


## 2. Check against eager PyTorch

The reference runs on CPU, regardless of the device selected for compilation. Both logits and bounding boxes are checked.


In [3]:
started = time.perf_counter()
with torch.inference_mode():
    expected = model(values)
eager_seconds = time.perf_counter() - started
print(f'Eager CPU call: {eager_seconds:.3f} s')
print({name: tuple(value.shape) for name, value in expected.items()})

device_setup_started = time.perf_counter()
if DEVICE == 'webgpu':
    from iree.runtime.webgpu import get_webgpu_device
    from iree.turbine.runtime.torch_device import TorchDevice
    gpu = TorchDevice(await get_webgpu_device(
        max_concurrent_operations=len(model.state_dict()) + 1), name='webgpu')
    backend = gpu.backend
else:
    import iree.turbine.runtime.torch_device
    gpu = None
    backend = 'turbine_cpu'

model = model.to(DEVICE)
device_input = values.to(DEVICE)
if gpu is not None:
    await gpu.synchronize()

device_setup_seconds = time.perf_counter() - device_setup_started
compiled_model = torch.compile(model, backend=backend, fullgraph=True, dynamic=False)
print(f'Compiling for {DEVICE}...', flush=True)
started = time.perf_counter()
with torch.inference_mode():
    answer = compiled_model(device_input)
outputs = {name: await gpu.to_cpu(answer[name]) if gpu is not None else answer[name].detach()
           for name in ('pred_logits', 'pred_boxes')}
cold_seconds = time.perf_counter() - started
for name in ('pred_logits', 'pred_boxes'):
    torch.testing.assert_close(outputs[name], expected[name], rtol=3e-3, atol=1e-3)
print('Compiled logits and boxes match eager PyTorch.')


Eager CPU call: 0.919 s
{'pred_logits': (1, 300, 80), 'pred_boxes': (1, 300, 4)}


/lib/python3.14/site-packages/torch/utils/_filelock.py:4: UserWarning: only soft file lock is available
  from filelock import FileLock as base_FileLock


Compiling for webgpu...
Compiled logits and boxes match eager PyTorch.


## 3. Measure inference

The first prediction includes compilation, execution and output copying.
Warmed predictions reuse resident inputs; output copying is measured separately.
These are inference measurements, not training or camera FPS.


In [4]:
repeats = 10
with torch.inference_mode():
    for _ in range(2):
        answer = compiled_model(device_input)
        if gpu is not None:
            await gpu.synchronize()
    completed_seconds = []
    for _ in range(repeats):
        started = time.perf_counter()
        answer = compiled_model(device_input)
        if gpu is not None:
            await gpu.synchronize()
        completed_seconds.append(time.perf_counter() - started)

started = time.perf_counter()
checked = {name: await gpu.to_cpu(answer[name]) if gpu is not None else answer[name].detach().clone()
           for name in ('pred_logits', 'pred_boxes')}
readback_seconds = time.perf_counter() - started
for name in ('pred_logits', 'pred_boxes'):
    torch.testing.assert_close(checked[name], expected[name], rtol=3e-3, atol=1e-3)
report = dict(model='D-FINE Small COCO', device=DEVICE, precision='FP32',
              model_input=list(values.shape), pytorch=torch.__version__,
              weights_and_model_load_seconds=weights_and_model_load_seconds,
              eager_cpu_reference_seconds=eager_seconds,
              device_setup_seconds=device_setup_seconds,
              compile_and_first_prediction_seconds=cold_seconds,
              resident_inference_median_ms=statistics.median(completed_seconds) * 1000,
              output_copy_or_readback_ms=readback_seconds * 1000,
              warmed_completed_seconds=completed_seconds, repeats=repeats,
              max_abs_error={name: float((checked[name] - expected[name]).abs().max())
                             for name in ('pred_logits', 'pred_boxes')}, passed=True)

display(Markdown(
    f"| {DEVICE.upper()} measurement | Time |\n| :--- | ---: |\n"
    f"| Download/cache and load model | {weights_and_model_load_seconds:.3f} s |\n"
    f"| Eager reference (CPU) | {eager_seconds:.3f} s |\n"
    f"| Device setup and initial upload | {device_setup_seconds:.3f} s |\n"
    f"| Compile and first prediction | {cold_seconds:.3f} s |\n"
    f"| Warmed prediction, resident input (median of {repeats}) | "
    f"{report['resident_inference_median_ms']:.2f} ms |\n"
    f"| Output copy or readback | {report['output_copy_or_readback_ms']:.2f} ms |"
))
print(json.dumps(report, indent=2))
_ = Path(f'benchmark-{DEVICE}.json').write_text(json.dumps(report, indent=2) + "\n")


| WEBGPU measurement | Time |
| :--- | ---: |
| Download/cache and load model | 0.283 s |
| Eager reference (CPU) | 0.919 s |
| Device setup and initial upload | 6.709 s |
| Compile and first prediction | 30.956 s |
| Warmed prediction, resident input (median of 10) | 363.20 ms |
| Output copy or readback | 3.60 ms |

{
  "model": "D-FINE Small COCO",
  "device": "webgpu",
  "precision": "FP32",
  "model_input": [
    1,
    3,
    640,
    640
  ],
  "pytorch": "2.15.0a0",
  "weights_and_model_load_seconds": 0.2828999999999997,
  "eager_cpu_reference_seconds": 0.9191000000000003,
  "device_setup_seconds": 6.7089,
  "compile_and_first_prediction_seconds": 30.955899999999996,
  "resident_inference_median_ms": 363.20000000000266,
  "output_copy_or_readback_ms": 3.5999999999987153,
  "warmed_completed_seconds": [
    0.3622000000000014,
    0.3592999999999975,
    0.3567999999999998,
    0.3567999999999998,
    0.3597999999999999,
    0.36420000000000385,
    0.39569999999999794,
    0.36769999999999925,
    0.3832999999999984,
    0.3896000000000015
  ],
  "repeats": 10,
  "max_abs_error": {
    "pred_logits": 0.000110626220703125,
    "pred_boxes": 3.2782554626464844e-06
  },
  "passed": true
}


## Recorded CPU and WebGPU comparison

Both runs use the same model, photo and FP32 inputs. Performance on your own
device may differ. The saved report includes machine and browser details.


In [7]:
record = json.loads(Path('benchmark-results.json').read_text())
cpu = record['compiled_inference']['cpu']
webgpu = record['compiled_inference']['webgpu']
display(Markdown(
    "| Measurement | CPU | WebGPU |\n| :--- | ---: | ---: |\n"
    f"| Compile and first prediction | {cpu['compile_and_first_prediction_seconds']:.2f} s | "
    f"{webgpu['compile_and_first_prediction_seconds']:.2f} s |\n"
    f"| Warmed prediction, resident input | {cpu['resident_inference_median_ms']:.2f} ms | "
    f"{webgpu['resident_inference_median_ms']:.2f} ms |\n"
    f"| Output copy or readback | {cpu['output_copy_or_readback_ms']:.2f} ms | "
    f"{webgpu['output_copy_or_readback_ms']:.2f} ms |"
))


| Measurement | CPU | WebGPU |
| :--- | ---: | ---: |
| Compile and first prediction | 23.14 s | 30.96 s |
| Warmed prediction, resident input | 1741.75 ms | 363.20 ms |
| Output copy or readback | 0.50 ms | 3.60 ms |